In [0]:
from pyspark.sql import Row

# Create dept DataFrame
dept_data = [
    Row(dept_id=i, dept_name=f"Dept_{i%10}", location=f"Location_{i%5}")
    for i in range(1, 11)
]
dept_df = spark.createDataFrame(dept_data)

# Create emp DataFrame with more than 50 records
emp_data = [
    Row(emp_id=i, emp_name=f"Emp_{i}", dept_id=(i % 10) + 1, salary=3000 + (i % 20) * 100)
    for i in range(1, 61)
]
emp_df = spark.createDataFrame(emp_data)

display(dept_df)
display(emp_df)

In [0]:
from pyspark.sql.functions import spark_partition_id
test=emp_df.repartition(4,"dept_id").withColumn("partiton_number",spark_partition_id())
display(test)

In [0]:
# display(dept_df)
# display(emp_df)

df_joined = emp_df.join(dept_df, on=[emp_df.dept_id == dept_df.dept_id], how="inner")
df_joined.select([emp_df.emp_name,emp_df.salary,dept_df.dept_name]).show()

In [0]:
e = emp_df.alias("e")
d = dept_df.alias("d")
df_joined = e.join(d, on=[e.dept_id == d.dept_id], how="inner")
display(df_joined.select([e.emp_name, e.salary, d.dept_name]))

In [0]:
df_joined = emp_df.alias("e").join(dept_df.alias("d"), on=[emp_df.dept_id == dept_df.dept_id], how="inner")
display(df_joined.select("e.emp_name", "e.salary", "d.dept_name"))

rdd.getpatitions
emp.coalesce 

In [0]:
from pyspark.sql.functions import col, when

# Example 1: Multiple join conditions (emp.dept_id == dept.dept_id AND emp.salary > 3100)
df_multi_cond = emp_df.join(
    dept_df,
    (emp_df.dept_id == dept_df.dept_id) & (emp_df.salary > 3100),
    how="inner"
)
display(df_multi_cond.select(emp_df.emp_name, emp_df.salary, dept_df.dept_name, dept_df.location))

# Example 2: Conditional join (CASCADE-like: join only if dept.location == 'Location_1' or emp.salary > 3500)
df_conditional = emp_df.join(
    dept_df,
    (emp_df.dept_id == dept_df.dept_id) & ((dept_df.location == "Location_1") | (emp_df.salary > 3500)),
    how="inner"
)
display(df_conditional.select(emp_df.emp_name, emp_df.salary, dept_df.dept_name, dept_df.location))

# Example 3: LEFT OUTER JOIN with additional filter after join
df_left_outer = emp_df.join(
    dept_df,
    emp_df.dept_id == dept_df.dept_id,
    how="left"
).filter(dept_df.location == "Location_2")
display(df_left_outer.select(emp_df.emp_name, emp_df.salary, dept_df.dept_name, dept_df.location))

# Example 4: Join multiple tables with different join types
# Create a bonus DataFrame for demonstration
bonus_data = [
    Row(emp_id=i, bonus=500 + (i % 5) * 100)
    for i in range(1, 61)
]
bonus_df = spark.createDataFrame(bonus_data)

# Inner join emp and dept, then left join bonus
df_multi_table = emp_df.join(
    dept_df,
    emp_df.dept_id == dept_df.dept_id,
    how="inner"
).join(
    bonus_df,
    emp_df.emp_id == bonus_df.emp_id,
    how="left"
)
display(df_multi_table.select(emp_df.emp_name, emp_df.salary, dept_df.dept_name, bonus_df.bonus))

# Example 5: Advanced join with computed column and conditional logic
df_advanced = emp_df.join(
    dept_df,
    emp_df.dept_id == dept_df.dept_id,
    how="inner"
).withColumn(
    "salary_grade",
    when(emp_df.salary > 3500, "High").when(emp_df.salary > 3200, "Medium").otherwise("Low")
)
display(df_advanced.select(emp_df.emp_name, emp_df.salary, dept_df.dept_name, dept_df.location, "salary_grade"))